# Generación — Notebook 03 (para Google Colab con GPU)
## Fine-tuning de **Stable Diffusion 1.5** con LoRA usando 🤗 diffusers

Este notebook aplica a Stable Diffusion 1.5 real el mismo procedimiento que el Notebook 02 aplicó a la versión en miniatura: cargar VAE, codificador CLIP, U-Net y scheduler pre-entrenados, congelarlos, añadir adaptadores LoRA a la atención de la U-Net y entrenarlos con la pérdida de difusión sobre un dataset imagen-texto.

* **Modelo base:** `stable-diffusion-v1-5/stable-diffusion-v1-5` (860 M parámetros en la U-Net).
* **Dataset:** `lambdalabs/naruto-blip-captions` (1 221 imágenes del anime *Naruto* con descripciones BLIP). Objetivo: que el modelo aprenda ese **estilo** de ilustración.
* **LoRA:** rango r = 8 en `to_q`, `to_k`, `to_v`, `to_out.0` (≈ 0.8 M parámetros entrenables, 0.1 % de la U-Net).
* **Hardware:** GPU T4 de Colab (16 GB). 500 pasos con lote 2 a 512×512 tardan ≈ 25–30 min.

> **Requisitos:** `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`. Algunos modelos de Hugging Face piden aceptar la licencia o iniciar sesión (`huggingface_hub.login()`).

In [ ]:
!pip install -q diffusers transformers accelerate peft datasets

In [ ]:
import os, math, time, json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import load_dataset
from torchvision import transforms
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, StableDiffusionPipeline
from transformers import CLIPTokenizer, CLIPTextModel
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict

assert torch.cuda.is_available(), "Active una GPU: Entorno de ejecución → Cambiar tipo de entorno → T4"
device, dtype = "cuda", torch.float16
MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
DATASET = "lambdalabs/naruto-blip-captions"
RES, BATCH, STEPS, LR, RANK = 512, 2, 500, 1e-4, 8
SEED = 42; torch.manual_seed(SEED)
for d in ("figures", "results", "modelos"): os.makedirs(d, exist_ok=True)
print(torch.cuda.get_device_name(0))

## 1. Componentes pre-entrenados de Stable Diffusion

In [ ]:
tokenizer = CLIPTokenizer.from_pretrained(MODEL, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(MODEL, subfolder="text_encoder", torch_dtype=dtype).to(device)
vae = AutoencoderKL.from_pretrained(MODEL, subfolder="vae", torch_dtype=dtype).to(device)
unet = UNet2DConditionModel.from_pretrained(MODEL, subfolder="unet", torch_dtype=dtype).to(device)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL, subfolder="scheduler")
for m in (text_encoder, vae, unet): m.requires_grad_(False)
count = lambda m: sum(p.numel() for p in m.parameters())
print(f"VAE {count(vae) / 1e6:.0f} M | CLIP {count(text_encoder) / 1e6:.0f} M | U-Net {count(unet) / 1e6:.0f} M | factor de escala VAE {vae.config.scaling_factor}")

## 2. Generación ANTES del fine-tuning

In [ ]:
PROMPTS = ["a man with blue hair and a red jacket", "a girl with green eyes smiling", "an old man with a white beard", "a cat wearing a hat"]
def make_pipe():
    p = StableDiffusionPipeline.from_pretrained(MODEL, unet=unet, text_encoder=text_encoder, vae=vae, tokenizer=tokenizer,
                                                torch_dtype=dtype, safety_checker=None, requires_safety_checker=False).to(device)
    p.set_progress_bar_config(disable=True); return p
def generate(pipe, prompts, seed=0, **kw):
    g = torch.Generator(device).manual_seed(seed)
    return pipe(prompts, num_inference_steps=30, guidance_scale=7.5, generator=g, **kw).images
pipe = make_pipe()
before = generate(pipe, PROMPTS)

## 3. Dataset y preprocesamiento
Redimensionado a 512 px, recorte central, espejo aleatorio y normalización a [−1, 1]; las descripciones se tokenizan con el tokenizador CLIP (77 tokens).

In [ ]:
ds = load_dataset(DATASET, split="train")
print(ds)
tf = transforms.Compose([transforms.Resize(RES), transforms.CenterCrop(RES), transforms.RandomHorizontalFlip(),
                         transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
def collate(batch):
    px = torch.stack([tf(b["image"].convert("RGB")) for b in batch])
    ids = tokenizer([b["text"] for b in batch], padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt").input_ids
    return px, ids
loader = torch.utils.data.DataLoader(ds, batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for a, i in zip(axes, range(4)):
    a.imshow(ds[i]["image"]); a.set_title(ds[i]["text"][:45], fontsize=8); a.axis("off")
plt.tight_layout(); plt.savefig("figures/03_dataset.png", bbox_inches="tight"); plt.show()

## 4. Adaptadores LoRA y bucle de entrenamiento
Mismo bucle que el script oficial `train_text_to_image_lora.py`: imagen → latente del VAE × factor de escala → ruido en un paso *t* aleatorio → la U-Net predice el ruido → MSE. Sólo se actualizan los adaptadores LoRA (en float32; el resto del modelo queda en float16 y congelado).

In [ ]:
unet.add_adapter(LoraConfig(r=RANK, lora_alpha=RANK, init_lora_weights="gaussian", target_modules=["to_k", "to_q", "to_v", "to_out.0"]))
lora_params = [p for p in unet.parameters() if p.requires_grad]
for p in lora_params: p.data = p.data.float()
print(f"Parámetros LoRA entrenables: {sum(p.numel() for p in lora_params):,} ({100 * sum(p.numel() for p in lora_params) / count(unet):.2f} % de la U-Net)")
unet.enable_gradient_checkpointing(); unet.train()
opt = torch.optim.AdamW(lora_params, lr=LR, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * s / STEPS)))
hist, step, t0 = [], 0, time.time()
while step < STEPS:
    for px, ids in loader:
        with torch.no_grad():
            latents = vae.encode(px.to(device, dtype)).latent_dist.sample() * vae.config.scaling_factor
            enc = text_encoder(ids.to(device))[0]
        noise = torch.randn_like(latents)
        t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=device)
        noisy = noise_scheduler.add_noise(latents, noise, t)
        with torch.autocast("cuda", dtype=dtype):
            pred = unet(noisy, t, encoder_hidden_states=enc).sample
        loss = F.mse_loss(pred.float(), noise.float())
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(lora_params, 1.0); opt.step(); sched.step()
        hist.append(loss.item()); step += 1
        if step % 50 == 0: print(f"paso {step:4d}/{STEPS} | pérdida {np.mean(hist[-50:]):.4f} | {time.time() - t0:.0f} s")
        if step >= STEPS: break
train_min = (time.time() - t0) / 60
unet.eval()
StableDiffusionPipeline.save_lora_weights("modelos/lora_naruto", unet_lora_layers=get_peft_model_state_dict(unet), safe_serialization=True)
plt.figure(figsize=(8, 3.4)); plt.plot(np.convolve(hist, np.ones(25) / 25, mode="valid")); plt.title("Pérdida de difusión (media móvil 25)"); plt.xlabel("paso")
plt.tight_layout(); plt.savefig("figures/03_perdida.png", bbox_inches="tight"); plt.show()

## 5. Generación DESPUÉS del fine-tuning y escala LoRA

In [ ]:
pipe = make_pipe()
after = generate(pipe, PROMPTS)
fig, axes = plt.subplots(2, 4, figsize=(16, 8.5))
for j, p in enumerate(PROMPTS):
    axes[0, j].imshow(before[j]); axes[0, j].set_title("ANTES\n" + p, fontsize=8)
    axes[1, j].imshow(after[j]); axes[1, j].set_title("DESPUÉS (LoRA)", fontsize=8)
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig("figures/03_antes_despues.png", bbox_inches="tight"); plt.show()

scales = [0.0, 0.5, 1.0]
row = [generate(pipe, [PROMPTS[0]], seed=1, cross_attention_kwargs={"scale": s})[0] for s in scales]
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
for a, s, im in zip(axes, scales, row): a.imshow(im); a.set_title(f"escala LoRA = {s}"); a.axis("off")
plt.tight_layout(); plt.savefig("figures/03_escala_lora.png", bbox_inches="tight"); plt.show()

## 6. Recarga del LoRA sobre el modelo base original

In [ ]:
base = StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=dtype, safety_checker=None, requires_safety_checker=False).to(device)
base.load_lora_weights("modelos/lora_naruto")
reload_img = generate(base, [PROMPTS[1]], seed=3)[0]
plt.imshow(reload_img); plt.axis("off"); plt.title("LoRA recargado: " + PROMPTS[1], fontsize=9); plt.savefig("figures/03_recarga.png", bbox_inches="tight"); plt.show()
json.dump({"pasos": STEPS, "lote": BATCH, "resolucion": RES, "rango": RANK, "minutos": train_min,
           "perdida_inicial": float(np.mean(hist[:25])), "perdida_final": float(np.mean(hist[-25:])),
           "lora_mb": os.path.getsize("modelos/lora_naruto/pytorch_lora_weights.safetensors") / 2**20}, open("results/03_sd15_lora.json", "w"), indent=2)